In [1]:
import pickle
import json
import pandas as pd

In [2]:
!unzip best_finetuned_dense_encoder.zip -d dense_encoder

Archive:  best_finetuned_dense_encoder.zip
   creating: dense_encoder/models/finder_dense_encoder_best/
  inflating: dense_encoder/models/finder_dense_encoder_best/config_sentence_transformers.json  
  inflating: dense_encoder/models/finder_dense_encoder_best/config.json  
  inflating: dense_encoder/models/finder_dense_encoder_best/model.safetensors  
  inflating: dense_encoder/models/finder_dense_encoder_best/tokenizer_config.json  
  inflating: dense_encoder/models/finder_dense_encoder_best/special_tokens_map.json  
  inflating: dense_encoder/models/finder_dense_encoder_best/vocab.txt  
  inflating: dense_encoder/models/finder_dense_encoder_best/tokenizer.json  
  inflating: dense_encoder/models/finder_dense_encoder_best/sentence_bert_config.json  
   creating: dense_encoder/models/finder_dense_encoder_best/1_Pooling/
  inflating: dense_encoder/models/finder_dense_encoder_best/1_Pooling/config.json  
   creating: dense_encoder/models/finder_dense_encoder_best/2_Normalize/
  inflating

In [7]:
!unzip ce_finder_best.zip -d ce_finder_best

Archive:  ce_finder_best.zip
   creating: /Users/kritishahi/Documents/Third Semester/NLP/Project/Final Codes/ce_finder_best/cross_encoder_finder_best
  inflating: ce_finder_best/cross_encoder_finder_best/README.md  
  inflating: ce_finder_best/cross_encoder_finder_best/tokenizer.json  
  inflating: ce_finder_best/cross_encoder_finder_best/vocab.txt  
  inflating: ce_finder_best/cross_encoder_finder_best/special_tokens_map.json  
  inflating: ce_finder_best/cross_encoder_finder_best/model.safetensors  
  inflating: ce_finder_best/cross_encoder_finder_best/tokenizer_config.json  
  inflating: ce_finder_best/cross_encoder_finder_best/config.json  


In [6]:
!pip install rank_bm25

In [5]:
!pip install sentence-transformers


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
with open("chunks_index.json", "r") as f:
    sec_chunks = json.load(f)

df = pd.read_csv("finder_augmented.csv")

df["company_name"] = df["company_name"].astype(str)


finder_queries = df.to_dict(orient="records")

finder_companies = set(df["company_name"].dropna().unique())
print(f"Found {len(finder_companies)} companies in FINdER dataset:", finder_companies)

filtered_chunks = [ch for ch in sec_chunks if ch["filename"] in finder_companies]

corpus_texts = [c["text"] for c in filtered_chunks]

Found 293 companies in FINdER dataset: {'NI', 'EBAY', 'DVN', 'EXPE', 'ABT', 'OKE', 'CDNS', 'APTV', 'TEL', 'AAPL', 'CEG', 'STLD', 'NVDA', 'IVZ', 'ABBV', 'MMM', 'ADI', 'OXY', 'PG', 'GOOGL', 'AVY', 'ETR', 'HON', 'NTRS', 'REGN', 'PCAR', 'TSCO', 'AWK', 'FFIV', 'MMC', 'LOW', 'DGX', 'CSGP', 'BDX', 'PFE', 'WEC', 'IEX', 'SJM', 'CMG', 'VLO,', 'SWK', 'MPC', 'ISRG', 'VLTO', 'JBHT', 'PH', 'GEN', 'LYB', 'AMP', 'ALL', 'SPG', 'PFG', 'META', 'HPQ', 'ACGL', 'TROW', 'AFL', 'BAC', 'AEE', 'PRU', 'BEN', 'ZBRA', 'APH', 'MCK', 'RL', 'MPWR', 'CARR', 'SRE', 'EA', 'KEYS', 'WBD', 'INTC', 'ODFL', 'VICI', 'REG', 'PNW', 'TXN', 'MCO', 'GNRC', 'TMUS', 'COO', 'NTAP', 'PLD', 'BKR', 'ATO', 'MOH', 'BBY', 'AMCR', 'GS', 'AES', 'TECH', 'PWR', 'PM', 'MRNA', 'TRMB', 'EXC', 'BX', 'NEM', 'AKAM', 'MOS', 'FITB', 'HCA', 'MDLZ', 'PTC', 'SMCI', 'NDAQ', 'ZBH', 'EVRG', 'BG', 'PSA', 'AMD', 'HII', 'PYPL', 'ROST', 'AXP', 'MKTX', 'SNPS', 'ADSK', 'CNP', 'JKHY', 'XYL', 'NSC', 'WMT', 'ARE', 'LVS', 'PNR', 'MTB', 'JPM', 'SPGI', 'MSCI', 'DG', 'C

In [3]:
from rank_bm25 import BM25Okapi

def build_bm25(corpus_texts, tokenizer):
    tokenized = [tokenizer(t) for t in corpus_texts]
    return BM25Okapi(tokenized)

In [4]:
import re
def normalize(s):
    return re.sub(r"\s+", " ", str(s).strip())
def tokenize(s):
    return normalize(s).lower().split()

bm25 = build_bm25(corpus_texts, tokenize)

In [6]:
from sentence_transformers import SentenceTransformer, CrossEncoder

dense_model = SentenceTransformer("dense_encoder/models/finder_dense_encoder_best/")
dense_model.max_seq_length = 256

In [7]:
import json

def load_finder_triplets(path):
    triplets = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            triplets.append({ "query": item["query"],
                "positive": item["positive"]["text"],
                "negatives": [n["text"] for n in item["negatives"]]
            })
    return triplets

triplets = load_finder_triplets("finder_triplets_optimized.jsonl")
print(f"Loaded {len(triplets)} evaluation queries")

Loaded 3439 evaluation queries


In [8]:
def convert_triplets_to_ids(triplets, text_to_id):
    converted = []
    skipped = 0

    for t in triplets:
        pos_text = t["positive"]

        if pos_text not in text_to_id:
            skipped += 1
            continue

        converted.append({
            "query": t["query"],
            "positive_id": text_to_id[pos_text]
        })

    print(f"Converted {len(converted)} triplets | Skipped {skipped}")
    return converted


In [11]:
text_to_id = {text: i for i, text in enumerate(corpus_texts)}

triplets_with_ids = convert_triplets_to_ids(triplets, text_to_id)

Converted 3439 triplets | Skipped 0


In [12]:
triplets

[{'query': 'Delta in CBOE Data & Access Solutions rev from 2021-23.',
  'positive': '31 2022 primarily due to increases in access and capacity fees and proprietary market data fees Access and capacity fees increased primarily due to increased physical port fees in the Options North American Equities and Europe and Asia Pacific segments and increased logical port fees in the Options North American Equities and Global FX segments both driven by an increase in subscribers and pricing Proprietary market data fees increased primarily due to an increase in proprietary market data fees in the Options segment coupled with an increase in proprietary market data fees attributable to Cboe Canada 83 Table of Contents Derivatives Markets Derivatives markets revenues less cost of revenues increased for the year ended December 31 2023 compared to the year ended December 31 2022 primarily due to increases in net transaction and clearing fees driven by a 33 increase in index options ADV partially offse

In [13]:
import torch
import numpy as np


def recall_at_k(ranks, k):
    return np.mean([r <= k for r in ranks])

def precision_at_k(ranks, k):
    return np.mean([1.0 / k if r <= k else 0.0 for r in ranks])

def mrr(ranks):
    return np.mean([1.0 / r if r != np.inf else 0.0 for r in ranks])

def ndcg_at_k(ranks, k):
    return np.mean([
        1.0 / np.log2(r + 1) if r <= k else 0.0
        for r in ranks
    ])

@torch.no_grad()
def corpus_level_eval_gpu(
    model,
    triplets,
    corpus_embeddings,
    ks=[5, 10, 20],
    batch_size=64,
    top_k=20,
    device="cuda"
):
    model.to(device)
    corpus_embeddings = corpus_embeddings.to(device)

    ranks = []

    queries = [t["query"] for t in triplets]
    positives = [t["positive_id"] for t in triplets]

    for i in range(0, len(queries), batch_size):
        batch_q = queries[i:i+batch_size]
        batch_pos = positives[i:i+batch_size]

        q_emb = model.encode(
            batch_q,
            convert_to_tensor=True,
            normalize_embeddings=True,
            device=device
        )

        scores = q_emb @ corpus_embeddings.T

        topk_idx = torch.topk(scores, k=top_k, dim=1).indices

        for row, pos_id in zip(topk_idx, batch_pos):
            row = row.tolist()
            if pos_id in row:
                ranks.append(row.index(pos_id) + 1)
            else:
                ranks.append(np.inf)

    metrics = {}
    for k in ks:
        metrics[f"Recall@{k}"] = recall_at_k(ranks, k)
        metrics[f"Precision@{k}"] = precision_at_k(ranks, k)
        metrics[f"nDCG@{k}"] = ndcg_at_k(ranks, k)

    metrics["MRR"] = mrr(ranks)
    metrics["Num_Queries"] = len(ranks)

    return metrics


In [15]:
with open('fine_tuned_sec_embeddings.pkl', 'rb') as f:
    corpus_embeddings = pickle.load(f)

In [16]:
metrics = corpus_level_eval_gpu(
    model=dense_model,
    triplets=triplets_with_ids,
    corpus_embeddings=corpus_embeddings,
    ks=[5, 10, 20],
    batch_size=64,
    top_k=200,
    device="cuda"
)

print(metrics)


{'Recall@5': np.float64(0.20587380052340798), 'Precision@5': np.float64(0.0411747601046816), 'nDCG@5': np.float64(0.13823619113618857), 'Recall@10': np.float64(0.281767955801105), 'Precision@10': np.float64(0.0281767955801105), 'nDCG@10': np.float64(0.16274643021408028), 'Recall@20': np.float64(0.37132887467287), 'Precision@20': np.float64(0.0185664437336435), 'nDCG@20': np.float64(0.18544103885380137), 'MRR': np.float64(0.1383357011825151), 'Num_Queries': 3439}


In [7]:
### If running on CPU then only it will be needed otherwise simply load the pickle file without class CPU_Unpickler
import pickle
import torch
import io

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)


with open('fine_tuned_sec_embeddings.pkl', 'rb') as f:
    corpus_embeddings = CPU_Unpickler(f).load()


In [8]:
import numpy as np

def recall_at_k(ranks, k):
    return np.mean([1 if r <= k else 0 for r in ranks])

def precision_at_k(ranks, k):
    return np.mean([1.0 / k if r <= k else 0.0 for r in ranks])

def mrr(ranks):
    return np.mean([1.0 / r if r != np.inf else 0.0 for r in ranks])

def ndcg_at_k(ranks, k):
    scores = []
    for r in ranks:
        if r <= k:
            scores.append(1.0 / np.log2(r + 1))
        else:
            scores.append(0.0)
    return np.mean(scores)


In [9]:
def dense_retrieve(query, model, corpus_embeddings, corpus_texts, top_k=200):
    q_emb = model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    scores = cos_sim(q_emb, corpus_embeddings)[0]
    top_idx = torch.topk(scores, top_k).indices.tolist()
    return [corpus_texts[i] for i in top_idx]


In [10]:
def bm25_retrieve(query, bm25, corpus_texts, top_k=200):
    scores = bm25.get_scores(tokenize(query))
    top_idx = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]
    return [corpus_texts[i] for i in top_idx]


In [11]:
def corpus_level_eval(
    triplets,
    retrieve_fn,
    ks=[5, 10, 20]
):
    ranks = []

    for t in triplets:
        query = t["query"]
        gold = t["positive"]

        retrieved = retrieve_fn(query)

        if gold in retrieved:
            rank = retrieved.index(gold) + 1
        else:
            rank = np.inf

        ranks.append(rank)

    metrics = {}

    for k in ks:
        metrics[f"Recall@{k}"] = recall_at_k(ranks, k)
        metrics[f"Precision@{k}"] = precision_at_k(ranks, k)
        metrics[f"nDCG@{k}"] = ndcg_at_k(ranks, k)

    metrics["MRR"] = mrr(ranks)

    return metrics


In [12]:
bm25_metrics = corpus_level_eval(triplets, retrieve_fn=lambda q: bm25_retrieve(q,bm25,corpus_texts,top_k=100))

print("BM25 Retrieval Metrics")
print(bm25_metrics)


BM25 Retrieval Metrics
{'Recall@5': 0.054085489968013954, 'Precision@5': 0.010817097993602792, 'nDCG@5': 0.037279230282647555, 'Recall@10': 0.0756033730735679, 'Precision@10': 0.00756033730735679, 'nDCG@10': 0.044293655744555224, 'Recall@20': 0.10031986042454202, 'Precision@20': 0.005015993021227101, 'nDCG@20': 0.050556706801230936, 'MRR': 0.0382900086911006}
